# Gold-standard sample for human validation

Selects a stratified sample of tweets from the master classified corpus,
to be hand-labelled by two human annotators in order to compute
inter-annotator agreement (Cohen's $\kappa$) and LLM-vs-human agreement
for both **topic** and **stance**.

## Design

- **Universe.** Full classified corpus, filtered to `usable_for_llm == True`
  and excluding any residual ERROR rows. The validation is about the
  *classifier*, so we sample over everything the classifier saw — not just
  the 4-actor comparative window.
- **Primary stratification.** `politician` × `topic` (4 × 7 = 28 cells).
  Proportional allocation with a per-cell floor to avoid empty strata.
- **Target N.** 400 tweets. Yields ≈57 per topic, sufficient for stable κ.
- **Rare-class oversample.** `stance == 'Unclear'` is heavily
  underrepresented (~5% under prompt v2). We add a top-up to guarantee
  ≥30 'Unclear' tweets in the final sample for stance κ.
- **Random seed.** 42 throughout (sampling, shuffling, file ordering).
- **Blinding.** Annotator files contain only `tweet_id`, `date`,
  `politician`, `tweet` and three empty fields
  (`annotator_topic`, `annotator_stance`, `annotator_notes`).
  The LLM labels are kept in a separate reference file that the
  annotators never see.

## Outputs

- `gold_standard_annotator_A.xlsx` — blind file, annotator A
- `gold_standard_annotator_B.xlsx` — blind file, annotator B
- `gold_standard_reference.csv` — same tweets with LLM labels (researcher only)
- `gold_standard_metadata.yaml` — seed, strata sizes, dates, model name
  for reproducibility

## 1. Setup

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

# ---- Config ---------------------------------------------------------------
INPUT_CSV = "classified_all_tweets_final_v2_clean.csv"

TARGET_N         = 400          # primary stratified sample
MIN_PER_CELL     = 4            # floor per (actor, topic) cell
TARGET_UNCLEAR   = 30           # minimum 'Unclear' tweets after oversample
SEED             = 42

OUT_ANNOTATOR_A  = "gold_standard_annotator_A2.xlsx"
OUT_ANNOTATOR_B  = "gold_standard_annotator_B2.xlsx"
OUT_REFERENCE    = "gold_standard_reference2.csv"
OUT_METADATA     = "gold_standard_metadata2.yaml"

TOPICS = [
    "Economy and Employment",
    "Welfare, Housing and Social Policy",
    "National Politics and Governance",
    "International Affairs",
    "Immigration and Security",
    "Rights and Equality",
    "Other",
]
STANCES = ["In favor", "Against", "Neutral", "Unclear"]

rng = np.random.default_rng(SEED)
print("Seed:", SEED)

Seed: 42


## 2. Load and filter

In [24]:
df = pd.read_csv(INPUT_CSV)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"Raw corpus: {len(df):,} rows")

# Drop ERROR rows and (if present) usable_for_llm == False
is_error = (df["topic"] == "ERROR") | (df["stance"] == "ERROR")
if "usable_for_llm" in df.columns:
    is_usable = df["usable_for_llm"].astype(bool)
else:
    is_usable = pd.Series(True, index=df.index)

pool = df[~is_error & is_usable].copy()
print(f"After dropping ERROR + non-usable: {len(pool):,} rows")

# Sanity: every actor present, dates parsed
assert pool["date"].notna().all(), "Some dates failed to parse"
print("\nActors:", sorted(pool["politician"].unique()))
print("Date range:", pool["date"].min().date(), "→", pool["date"].max().date())

Raw corpus: 11,866 rows
After dropping ERROR + non-usable: 11,866 rows

Actors: ['NunezFeijoo', 'Santi_ABASCAL', 'Yolanda_Diaz_', 'sanchezcastejon']
Date range: 2022-04-23 → 2026-04-22


## 3. Distribution by actor × topic

Inspect cell sizes before allocating quotas — strata too small to meet
the floor will be capped at their actual size.

In [25]:
actor_topic = (
    pool.groupby(["politician", "topic"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=TOPICS, fill_value=0)
)
actor_topic["total"] = actor_topic.sum(axis=1)
print(actor_topic.to_string())

topic            Economy and Employment  Welfare, Housing and Social Policy  National Politics and Governance  International Affairs  Immigration and Security  Rights and Equality  Other  total
politician                                                                                                                                                                                       
NunezFeijoo                         279                                 204                              1559                    254                        70                   82    441   2889
Santi_ABASCAL                       129                                  61                              1278                    323                       307                   63    195   2356
Yolanda_Diaz_                      1139                                 377                               837                    328                        26                  348    492   3547
sanchezcastejon               

## 4. Stratified allocation

**Rule:** for each (actor, topic) cell, draw

$$ n_{\text{cell}} = \max\!\bigl(\text{MIN\_PER\_CELL},\ 
    \bigl\lfloor \text{TARGET\_N} \cdot \frac{N_{\text{cell}}}{N_{\text{pool}}} \bigr\rfloor \bigr) $$

capped at the cell's actual size. We then trim the largest cells to
land exactly on `TARGET_N`.

In [26]:
N_total = len(pool)
actor_topic_counts = actor_topic.drop(columns="total")

# Initial proportional + floor
alloc = actor_topic_counts.copy().astype(int)
for actor in alloc.index:
    for topic in alloc.columns:
        n_cell = actor_topic_counts.loc[actor, topic]
        if n_cell == 0:
            alloc.loc[actor, topic] = 0
        else:
            prop = int(np.floor(TARGET_N * n_cell / N_total))
            alloc.loc[actor, topic] = min(max(MIN_PER_CELL, prop), n_cell)

current_total = int(alloc.values.sum())
print(f"After floor + proportional: {current_total} (target {TARGET_N})")

# Adjust to hit exactly TARGET_N — trim from largest non-floor cells if over,
# add to largest cells with available headroom if under.
def adjust(alloc, actor_topic_counts, target):
    diff = target - int(alloc.values.sum())
    while diff != 0:
        if diff < 0:
            # Reduce: pick the largest cell currently above the floor
            mask = alloc > MIN_PER_CELL
            if not mask.any().any():
                break
            ai, ti = np.unravel_index(
                np.argmax(np.where(mask, alloc.values, -1)), alloc.shape
            )
            alloc.iat[ai, ti] -= 1
            diff += 1
        else:
            # Increase: pick the cell with most headroom
            headroom = actor_topic_counts.values - alloc.values
            if headroom.max() <= 0:
                break
            ai, ti = np.unravel_index(np.argmax(headroom), alloc.shape)
            alloc.iat[ai, ti] += 1
            diff -= 1
    return alloc

alloc = adjust(alloc, actor_topic_counts, TARGET_N)
print(f"\nFinal allocation matrix (sum = {int(alloc.values.sum())}):")
print(alloc.to_string())

After floor + proportional: 402 (target 400)

Final allocation matrix (sum = 400):
topic            Economy and Employment  Welfare, Housing and Social Policy  National Politics and Governance  International Affairs  Immigration and Security  Rights and Equality  Other
politician                                                                                                                                                                                
NunezFeijoo                           9                                   6                                50                      8                         4                    4     14
Santi_ABASCAL                         4                                   4                                43                     10                        10                    4      6
Yolanda_Diaz_                        38                                  12                                28                     11                         4           

## 5. Draw the primary stratified sample

In [27]:
primary_chunks = []
for actor in alloc.index:
    for topic in alloc.columns:
        n_cell = int(alloc.loc[actor, topic])
        if n_cell == 0:
            continue
        cell_pool = pool[(pool["politician"] == actor) & (pool["topic"] == topic)]
        if len(cell_pool) == 0:
            continue
        sampled = cell_pool.sample(n=n_cell, random_state=SEED)
        primary_chunks.append(sampled)

primary = pd.concat(primary_chunks, ignore_index=True)
print(f"Primary sample drawn: {len(primary)} tweets")
print("\nPrimary sample stance distribution:")
print(primary["stance"].value_counts().to_string())

Primary sample drawn: 400 tweets

Primary sample stance distribution:
stance
In favor    253
Against      99
Neutral      33
Unclear      15


## 6. Top-up rare stance ('Unclear')

If the primary sample contains fewer than `TARGET_UNCLEAR` tweets labelled
as 'Unclear', draw additional ones from the pool, stratified by actor.

In [28]:
n_unclear_now = (primary["stance"] == "Unclear").sum()
needed = max(0, TARGET_UNCLEAR - n_unclear_now)
print(f"'Unclear' in primary: {n_unclear_now}; need {needed} more.")

if needed > 0:
    # Stratify the top-up by actor — proportional to each actor's stock of Unclear
    used_ids = set(primary["tweet_id"])
    unclear_pool = pool[(pool["stance"] == "Unclear") & (~pool["tweet_id"].isin(used_ids))]
    print(f"Available 'Unclear' outside primary: {len(unclear_pool)}")

    quotas = (
        unclear_pool.groupby("politician").size()
        .pipe(lambda s: (s / s.sum() * needed).round().astype(int))
    )
    # Adjust rounding to hit `needed` exactly
    while quotas.sum() < needed:
        quotas[quotas.idxmin()] += 1
    while quotas.sum() > needed:
        quotas[quotas.idxmax()] -= 1
    print(f"Top-up quotas by actor:\n{quotas.to_string()}")

    topup_chunks = []
    for actor, q in quotas.items():
        if q == 0:
            continue
        cell = unclear_pool[unclear_pool["politician"] == actor]
        topup_chunks.append(cell.sample(n=min(q, len(cell)), random_state=SEED))
    topup = pd.concat(topup_chunks, ignore_index=True) if topup_chunks else pd.DataFrame()
    print(f"Top-up drawn: {len(topup)} tweets")
else:
    topup = pd.DataFrame()

'Unclear' in primary: 15; need 15 more.
Available 'Unclear' outside primary: 516
Top-up quotas by actor:
politician
NunezFeijoo        2
Santi_ABASCAL      5
Yolanda_Diaz_      6
sanchezcastejon    2
Top-up drawn: 15 tweets


## 7. Build final sample, shuffle, and tag origin

In [ ]:
primary_tagged = primary.assign(sample_origin="stratified")
topup_tagged = topup.assign(sample_origin="unclear_topup") if len(topup) else topup

if len(topup_tagged):
    final = pd.concat([primary_tagged, topup_tagged], ignore_index=True)
else:
    final = primary_tagged.copy()

final = final.drop_duplicates(subset="tweet_id").sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

final["sample_order"] = np.arange(1, len(final) + 1)

# Restrict annotation file to the first 200 tweets
ANNOTATION_N = 200
final = final[final["sample_order"] <= ANNOTATION_N].copy()

# Reset sample_order after trimming
final["sample_order"] = np.arange(1, len(final) + 1)

print(f"Final sample size: {len(final)}")
print("\nDistribution by actor:")
print(final["politician"].value_counts().to_string())
print("\nDistribution by topic:")
print(final["topic"].value_counts().to_string())
print("\nDistribution by stance:")
print(final["stance"].value_counts().to_string())
print("\nDistribution by sample origin:")
print(final["sample_origin"].value_counts().to_string())

Final sample size: 415

Distribution by actor:
politician
Yolanda_Diaz_      126
sanchezcastejon    106
NunezFeijoo         97
Santi_ABASCAL       86

Distribution by topic:
topic
National Politics and Governance      149
Economy and Employment                 67
Other                                  60
International Affairs                  57
Welfare, Housing and Social Policy     32
Rights and Equality                    28
Immigration and Security               22

Distribution by stance:
stance
In favor    253
Against      99
Neutral      33
Unclear      30

Distribution by sample origin:
sample_origin
stratified       400
unclear_topup     15


## 8. Build blind annotator files (XLSX with dropdowns)

Same tweets, same order, in two separate files. Annotators must work
independently. Dropdowns prevent typos and constrain values to the schema.

In [30]:
BLIND_COLS = ["sample_order", "tweet_id", "date", "politician", "tweet"]
ANNOTATOR_COLS = ["annotator_topic", "annotator_stance", "annotator_notes"]

blind = final[BLIND_COLS].copy()
blind["date"] = blind["date"].dt.date.astype(str)
for c in ANNOTATOR_COLS:
    blind[c] = ""

def write_annotator_xlsx(df_blind: pd.DataFrame, path: str, annotator_id: str):
    wb = Workbook()
    ws = wb.active
    ws.title = f"Annotator_{annotator_id}"

    # Hidden sheet with allowed labels
    ws_labels = wb.create_sheet("labels")

    for i, topic in enumerate(TOPICS, start=1):
        ws_labels.cell(row=i, column=1, value=topic)

    for i, stance in enumerate(STANCES, start=1):
        ws_labels.cell(row=i, column=2, value=stance)

    ws_labels.sheet_state = "hidden"

    cols = list(df_blind.columns)
    ws.append(cols)

    for _, row in df_blind.iterrows():
        ws.append(list(row.values))

    # Column widths
    widths = {
        "sample_order": 12,
        "tweet_id": 22,
        "date": 12,
        "politician": 18,
        "tweet": 80,
        "annotator_topic": 40,
        "annotator_stance": 18,
        "annotator_notes": 40,
    }

    for i, c in enumerate(cols, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(c, 18)

    # Wrap text in tweet column
    tweet_col = get_column_letter(cols.index("tweet") + 1)
    for cell in ws[tweet_col][1:]:
        cell.alignment = cell.alignment.copy(wrap_text=True)

    n_rows = len(df_blind) + 1
    topic_col = get_column_letter(cols.index("annotator_topic") + 1)
    stance_col = get_column_letter(cols.index("annotator_stance") + 1)

    # Use ranges instead of comma-separated lists
    dv_topic = DataValidation(
        type="list",
        formula1="=labels!$A$1:$A$7",
        allow_blank=True,
        showErrorMessage=True,
        errorTitle="Invalid topic",
        error="Pick a value from the dropdown.",
    )

    dv_stance = DataValidation(
        type="list",
        formula1="=labels!$B$1:$B$4",
        allow_blank=True,
        showErrorMessage=True,
        errorTitle="Invalid stance",
        error="Pick a value from the dropdown.",
    )

    ws.add_data_validation(dv_topic)
    ws.add_data_validation(dv_stance)

    dv_topic.add(f"{topic_col}2:{topic_col}{n_rows}")
    dv_stance.add(f"{stance_col}2:{stance_col}{n_rows}")

    ws.freeze_panes = "A2"

    wb.save(path)
    print(f"Wrote {path}")

write_annotator_xlsx(blind, OUT_ANNOTATOR_A, "A")
write_annotator_xlsx(blind, OUT_ANNOTATOR_B, "B")

C:\Users\narci\AppData\Local\Temp\ipykernel_25216\620128053.py:49: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(wrap_text=True)


Wrote gold_standard_annotator_A.xlsx


C:\Users\narci\AppData\Local\Temp\ipykernel_25216\620128053.py:49: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(wrap_text=True)


Wrote gold_standard_annotator_B.xlsx


## 9. Reference file (researcher only)

Same tweets and order, plus the LLM labels — never shared with annotators.
Used after annotation to compute agreement metrics.

In [31]:
REF_COLS = BLIND_COLS + [
    "topic", "stance", "confidence", "short_justification", "sample_origin"
]
REF_COLS = [c for c in REF_COLS if c in final.columns]
reference = final[REF_COLS].copy()
reference = reference.rename(
    columns={
        "topic": "llm_topic",
        "stance": "llm_stance",
        "confidence": "llm_confidence",
        "short_justification": "llm_justification",
    }
)
reference.to_csv(OUT_REFERENCE, index=False, encoding="utf-8-sig")
print(f"Wrote {OUT_REFERENCE}  ({len(reference)} rows)")

Wrote gold_standard_reference.csv  (415 rows)


## 10. Sampling metadata

YAML file documenting every choice that defines this sample, so it can
be re-derived bit-for-bit later. Cite this in the TFM methodology section.

In [32]:
metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "input_csv": INPUT_CSV,
    "target_n": TARGET_N,
    "min_per_cell": MIN_PER_CELL,
    "target_unclear": TARGET_UNCLEAR,
    "random_seed": SEED,
    "final_n": int(len(final)),
    "unclear_in_final": int((final["stance"] == "Unclear").sum()),
    "corpus_pool_n_after_filters": int(len(pool)),
    "date_range": {
        "min": str(pool["date"].min().date()),
        "max": str(pool["date"].max().date()),
    },
    "actors": sorted(pool["politician"].unique().tolist()),
    "topics": TOPICS,
    "stances": STANCES,
    "allocation_matrix": alloc.astype(int).to_dict(),
    "final_distribution": {
        "by_actor": final["politician"].value_counts().to_dict(),
        "by_topic": final["topic"].value_counts().to_dict(),
        "by_stance": final["stance"].value_counts().to_dict(),
        "by_origin": final["sample_origin"].value_counts().to_dict(),
    },
    "output_files": {
        "annotator_A": OUT_ANNOTATOR_A,
        "annotator_B": OUT_ANNOTATOR_B,
        "reference": OUT_REFERENCE,
    },
}

with open(OUT_METADATA, "w", encoding="utf-8") as f:
    yaml.safe_dump(metadata, f, sort_keys=False, allow_unicode=True)
print(f"Wrote {OUT_METADATA}")

Wrote gold_standard_metadata.yaml


## 11. Final summary

In [33]:
print("=" * 72)
print("GOLD-STANDARD SAMPLE — SUMMARY")
print("=" * 72)
print(f"Input corpus       : {INPUT_CSV}")
print(f"Pool after filters : {len(pool):,} tweets")
print(f"Final sample size  : {len(final)} tweets")
print(f"Unclear oversample : {len(topup) if isinstance(topup, pd.DataFrame) else 0} tweets")
print(f"Random seed        : {SEED}")
print()
print("Files written:")
for p in [OUT_ANNOTATOR_A, OUT_ANNOTATOR_B, OUT_REFERENCE, OUT_METADATA]:
    size_kb = Path(p).stat().st_size / 1024 if Path(p).exists() else None
    print(f"  {p}  ({size_kb:.1f} KB)" if size_kb else f"  {p}  (missing)")
print()
print("Next steps:")
print("  1. Hand the two .xlsx files to annotators A and B.")
print("  2. Annotators fill `annotator_topic`, `annotator_stance`,")
print("     `annotator_notes` independently.")
print("  3. After annotation, merge by `tweet_id` against the reference")
print("     to compute Cohen's kappa (annotator A vs B, and consensus vs LLM).")

GOLD-STANDARD SAMPLE — SUMMARY
Input corpus       : classified_all_tweets_final_v2_clean.csv
Pool after filters : 11,866 tweets
Final sample size  : 415 tweets
Unclear oversample : 15 tweets
Random seed        : 42

Files written:
  gold_standard_annotator_A.xlsx  (70.9 KB)
  gold_standard_annotator_B.xlsx  (70.9 KB)
  gold_standard_reference.csv  (159.4 KB)
  gold_standard_metadata.yaml  (2.1 KB)

Next steps:
  1. Hand the two .xlsx files to annotators A and B.
  2. Annotators fill `annotator_topic`, `annotator_stance`,
     `annotator_notes` independently.
  3. After annotation, merge by `tweet_id` against the reference
     to compute Cohen's kappa (annotator A vs B, and consensus vs LLM).
